In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class SimpleEmbeddingStore:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)
        display(self.model)
        self.documents = []
        self.embeddings = np.empty((0, self.model.get_sentence_embedding_dimension())) # initialize with empty embeddings

    def add(self, documents):        
        new_embs = self.model.encode(documents)
        # documents & embeddings are simply paired by index, stored in the same order
        self.documents.extend(documents)
        self.embeddings = np.vstack([self.embeddings, new_embs]) if self.embeddings.size else new_embs # add new embeddings to store

    def encode(self, document: str):
        return self.model.encode(document)

    def decode(self, embedding):
        sims = cosine_similarity([embedding], self.embeddings)[0]
        print(sims)
        return self.documents[int(np.argmax(sims))] # lookup/retrieval mechanism

lookup = SimpleEmbeddingStore("Qwen/Qwen3-Embedding-0.6B")
seed_documents = [
    "The weather is sunny today.",
    "Neural networks power many AI applications.",
    "Coffee tastes best right after brewing."
]
lookup.add(seed_documents)
print(f"Store initialized: {lookup.embeddings.shape} embeddings\n")

encoded = lookup.encode("How to make a perfect cup of espresso?")
best_matched_document = lookup.decode(encoded)
print(best_matched_document)

encoded = lookup.encode("Is today a good day for a walk in the park?")
best_matched_document = lookup.decode(encoded)
print(best_matched_document)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 32768, 'do_lower_case': False, 'architecture': 'Qwen3Model'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': True, 'include_prompt': True})
  (2): Normalize()
)

Store initialized: (3, 1024) embeddings

[0.3061018  0.21463099 0.59308636]
Coffee tastes best right after brewing.
[0.58156955 0.3260554  0.37560374]
The weather is sunny today.
